# 🧬 ConsortiumTuner v3 — 6 modules, 20 circuits, live simulation

**BE552 Project**

---

## What's new in v3

- **Exactly 20 canonical circuits** — names match `whole_ecosystem_20circuits.xml` (the SBML ecosystem model in iBioSim)
- **6 modules** (not 7) — nitrification and denitrification collapsed into a single **Nitrogen cycle** module, matching the SBML
- **Genetic parts reference** — a read-only table showing the actual GOLDBAR parts (promoter / RBS / CDS / terminator) for each of the 20 circuits, grouped by module
- **One big iBioSim-style plot** with a species picker (presets + 33 individual checkboxes, just like iBioSim's *Edit Graph → run-1* dialog)
- **Live re-integration** — the moment you move a slider, the ecosystem re-integrates and the trajectories redraw

## The 6 modules

| # | Module | Circuits in library |
|---|---|---|
| 1 | **Cross-feeding** | Ecoli_K · Ecoli_L · Ecoli_Trp · Ecoli_His |
| 2 | **Nitrogen cycle** | Nitrosomonas · Nitrobacter · Pseudomonas_stutzeri · Paracoccus_denitrificans |
| 3 | **Electrogenesis** | StrainA_Gsulfurreducens · StrainB_Gmetalireducens · OmcZ_Booster_StrainA · OmcZ_Booster_StrainB |
| 4 | **Quorum sensing** | Luxl_LuxR · LuxS |
| 5 | **Kill switch** | KillSwitch_StrainA · KillSwitch_StrainB |
| 6 | **Metabolite production** | BetaCarotene · Isoprenoid · Naringenin · MalonylCoA |

Each slider scales the **rate constants** of its module between 0 (module off) and 1 (module at native strength). The selected consortium + live dynamics update together.


## Step 1 — Imports

In [1]:
# ---------------------------------------------------------------------------
# Standard scientific-Python stack plus Jupyter widgets for the interactive UI.
# ---------------------------------------------------------------------------
import ipywidgets as widgets                              # sliders, layout containers, observable widgets
from IPython.display import display, clear_output, Markdown  # in-notebook rendering helpers
import matplotlib.pyplot as plt                           # time-series and fitness bar plots
import numpy as np                                        # vector math + initial-condition array
from scipy.integrate import solve_ivp                     # ODE solver (LSODA) used in Step 4

# Bump default DPI so inline figures look crisp on retina/HiDPI displays.
plt.rcParams['figure.dpi'] = 100
print("Imports OK")


Imports OK


## Step 2 — Circuit library (20 circuits, 6 modules)

Every circuit scores `1.0` on its module axis and `0.0` elsewhere. Dependencies
(e.g. `KillSwitch_StrainA` requires `StrainA_Gsulfurreducens`) are enforced
in the selection step.


In [2]:
# ============================================================================
# Module names. Order here is the canonical order used by every downstream
# component (sliders, bar chart, color map, SBML reference). Must match the
# six module keys in whole_ecosystem_20circuits.xml.
# ============================================================================
MODULES = [
    'cross_feeding', 'nitrogen_cycle', 'electrogenesis',
    'quorum_sensing', 'kill_switch', 'metabolite_production'
]

def _scores(active):
    """Build a one-hot score vector for a circuit.

    Each circuit belongs to exactly one module, so its score vector is 1.0
    on that module's axis and 0.0 on the other five. The selection step in
    Step 5 takes the dot product of this vector with the slider weights to
    decide which circuits make it into the consortium.
    """
    return {m: (1.0 if m == active else 0.0) for m in MODULES}

# ============================================================================
# 20-circuit library. Each entry mirrors a circuit in the SBML ecosystem
# model. Schema per circuit:
#   organism : host strain / chassis (display only)
#   module   : which of the 6 modules — drives scoring & color
#   spec     : GOLDBAR-style genetic spec (overwritten from CIRCUIT_PARTS in Step 3)
#   note     : short biological description shown in the consortium table
#   scores   : one-hot vector across MODULES (produced by _scores())
#   requires : circuits that must also be selected if this one is. The kill
#              switches, for example, need their host strain to be present.
#              Enforced by dependency closure in select_consortium().
# ============================================================================
CIRCUITS = {
    # ===== 1. CROSS-FEEDING (4) =====
    # Two amino-acid trade pairs: Lys<->Leu and Trp<->His. Each strain
    # over-produces one AA and is auxotrophic for another, forcing mutualism.
    'Ecoli_K': {
        'organism': 'Escherichia coli (lys+ / leu-)',
        'module': 'cross_feeding',
        'spec': 'Ptrc then RBS2 then lysC then argD then lysA then T2',
        'note': 'Lysine producer, Leu auxotroph',
        'scores': _scores('cross_feeding'), 'requires': []
    },
    'Ecoli_L': {
        'organism': 'Escherichia coli (leu+ / lys-)',
        'module': 'cross_feeding',
        'spec': 'Ptrc then RBS1 then leuB then leuC then leuD then T1',
        'note': 'Leucine producer, Lys auxotroph',
        'scores': _scores('cross_feeding'), 'requires': []
    },
    'Ecoli_Trp': {
        'organism': 'Escherichia coli (trp+ / his-)',
        'module': 'cross_feeding',
        'spec': 'Ptrc then RBS1 then trpB then trpA then T1',
        'note': 'Tryptophan producer, His auxotroph',
        'scores': _scores('cross_feeding'), 'requires': []
    },
    'Ecoli_His': {
        'organism': 'Escherichia coli (his+ / trp-)',
        'module': 'cross_feeding',
        'spec': 'Ptrc then RBS1 then hisC then hisD then T1',
        'note': 'Histidine producer, Trp auxotroph',
        'scores': _scores('cross_feeding'), 'requires': []
    },

    # ===== 2. NITROGEN CYCLE (4) =====
    # Full nitrification + denitrification cascade: NH4 -> NO2 -> NO3 -> NO
    # -> N2O -> N2. Two nitrifiers feed two denitrifiers; downstream steps
    # require their upstream partners to be present (see `requires`).
    'Nitrosomonas': {
        'organism': 'Nitrosomonas europaea',
        'module': 'nitrogen_cycle',
        'spec': 'Ptrc then RBS1 then amoA then hao then T1',
        'note': 'NH4+ -> NO2- (amoA, hao)',
        'scores': _scores('nitrogen_cycle'), 'requires': []
    },
    'Nitrobacter': {
        'organism': 'Nitrobacter winogradskyi',
        'module': 'nitrogen_cycle',
        'spec': 'Ptrc then RBS2 then nxrA then T2',
        'note': 'NO2- -> NO3- (nxrA)',
        'scores': _scores('nitrogen_cycle'), 'requires': ['Nitrosomonas']
    },
    'Pseudomonas_stutzeri': {
        'organism': 'Pseudomonas stutzeri',
        'module': 'nitrogen_cycle',
        'spec': 'Ptrc then RBS1 then narG then nirS then T1',
        'note': 'NO3- -> NO2- -> NO (early denitrification)',
        'scores': _scores('nitrogen_cycle'), 'requires': []
    },
    'Paracoccus_denitrificans': {
        'organism': 'Paracoccus denitrificans',
        'module': 'nitrogen_cycle',
        'spec': 'Ptrc then RBS2 then norB then nosZ then T2',
        'note': 'NO -> N2O -> N2 (late denitrification)',
        'scores': _scores('nitrogen_cycle'), 'requires': ['Pseudomonas_stutzeri']
    },

    # ===== 3. ELECTROGENESIS (4) =====
    # Two Geobacter chassis (Strain A = G. sulfurreducens, Strain B =
    # G. metallireducens) plus optional OmcZ-overexpression boosters that
    # require the corresponding parent strain to already be in the consortium.
    'StrainA_Gsulfurreducens': {
        'organism': 'Geobacter sulfurreducens',
        'module': 'electrogenesis',
        'spec': 'Ptrc then RBS1 then acs then Term1',
        'note': 'acs expression; anode-reducing chassis',
        'scores': _scores('electrogenesis'), 'requires': []
    },
    'StrainB_Gmetalireducens': {
        'organism': 'Geobacter metallireducens',
        'module': 'electrogenesis',
        'spec': 'PnirH then RBS2 then omcZ then Term2',
        'note': 'PnirH-driven omcZ; NO-responsive EET',
        'scores': _scores('electrogenesis'), 'requires': []
    },
    'OmcZ_Booster_StrainA': {
        'organism': 'Geobacter sulfurreducens',
        'module': 'electrogenesis',
        'spec': 'PomcZ then RBS1 then omcZ then Term1',
        'note': 'Extra PomcZ->omcZ cassette in Strain A',
        'scores': _scores('electrogenesis'), 'requires': ['StrainA_Gsulfurreducens']
    },
    'OmcZ_Booster_StrainB': {
        'organism': 'Geobacter metallireducens',
        'module': 'electrogenesis',
        'spec': 'PomcZ then RBS2 then omcZ then Term2',
        'note': 'Extra PomcZ->omcZ cassette in Strain B',
        'scores': _scores('electrogenesis'), 'requires': ['StrainB_Gmetalireducens']
    },

    # ===== 4. QUORUM SENSING (2) =====
    # LuxI/LuxR makes & senses AHL (drives the kill-switch trigger via
    # KillSignal); LuxS makes the universal interspecies signal AI-2.
    'Luxl_LuxR': {
        'organism': 'LuxI/LuxR-carrying chassis',
        'module': 'quorum_sensing',
        'spec': 'Plux then RBS1 then luxI then luxR then T1',
        'note': 'AHL production + sensing (drives KillSignal)',
        'scores': _scores('quorum_sensing'), 'requires': []
    },
    'LuxS': {
        'organism': 'LuxS-carrying chassis',
        'module': 'quorum_sensing',
        'spec': 'Ptrc then RBS1 then luxS then T1',
        'note': 'AI-2 universal interspecies QS signal',
        'scores': _scores('quorum_sensing'), 'requires': []
    },

    # ===== 5. KILL SWITCH (2) =====
    # Toxin-antitoxin (MazE/MazF) cassettes — one per Geobacter strain.
    # Composite circuits: each is two transcriptional units (antitoxin first,
    # then toxin under a different promoter). Both require their host strain.
    'KillSwitch_StrainA': {
        'organism': 'Geobacter sulfurreducens',
        'module': 'kill_switch',
        'spec': '(Pconst then RBS1 then mazE then Term1) then (Prep then RBS1 then mazF then Term1)',
        'note': 'MazE/MazF toxin-antitoxin in Strain A',
        'scores': _scores('kill_switch'), 'requires': ['StrainA_Gsulfurreducens']
    },
    'KillSwitch_StrainB': {
        'organism': 'Geobacter metallireducens',
        'module': 'kill_switch',
        'spec': '(Pconst then RBS2 then mazE then Term2) then (Prep then RBS2 then mazF then Term2)',
        'note': 'Orthogonal MazE/MazF in Strain B',
        'scores': _scores('kill_switch'), 'requires': ['StrainB_Gmetalireducens']
    },

    # ===== 6. METABOLITE PRODUCTION (4) =====
    # Linear pathway: AcetylCoA -> MalonylCoA -> Naringenin and
    # AcetylCoA -> Isoprenoid -> BetaCarotene. Each circuit is one branch.
    'BetaCarotene': {
        'organism': 'Engineered carotenoid producer',
        'module': 'metabolite_production',
        'spec': 'Ptrc then RBS1 then crtE then crtB then crtI then T1',
        'note': 'Isoprenoid -> BetaCarotene',
        'scores': _scores('metabolite_production'), 'requires': []
    },
    'Isoprenoid': {
        'organism': 'Engineered isoprenoid producer',
        'module': 'metabolite_production',
        'spec': 'Ptrc then RBS1 then atoB then HMGS then HMGR then T1',
        'note': 'AcetylCoA -> Isoprenoid (mevalonate pathway)',
        'scores': _scores('metabolite_production'), 'requires': []
    },
    'Naringenin': {
        'organism': 'Engineered flavonoid producer',
        'module': 'metabolite_production',
        'spec': 'Ptrc then RBS1 then TAL then 4CL then CHS then CHI then T1',
        'note': 'MalonylCoA -> Naringenin',
        'scores': _scores('metabolite_production'), 'requires': []
    },
    'MalonylCoA': {
        'organism': 'Engineered malonyl-CoA producer',
        'module': 'metabolite_production',
        'spec': 'PfapR then RBS1 then fapR then fabH then T1',
        'note': 'AcetylCoA -> MalonylCoA (fapR/fabH)',
        'scores': _scores('metabolite_production'), 'requires': []
    },
}

# Sanity check — fail loudly if any circuit was added/removed accidentally.
assert len(CIRCUITS) == 20, f"Expected 20 circuits, got {len(CIRCUITS)}"

# Print a per-module summary so the user can confirm the library loaded right.
print(f"Loaded {len(CIRCUITS)} circuits across {len(MODULES)} modules:")
for m in MODULES:
    members = [n for n,c in CIRCUITS.items() if c['module'] == m]
    print(f"  {m:25s} ({len(members)}): {', '.join(members)}")


Loaded 20 circuits across 6 modules:
  cross_feeding             (4): Ecoli_K, Ecoli_L, Ecoli_Trp, Ecoli_His
  nitrogen_cycle            (4): Nitrosomonas, Nitrobacter, Pseudomonas_stutzeri, Paracoccus_denitrificans
  electrogenesis            (4): StrainA_Gsulfurreducens, StrainB_Gmetalireducens, OmcZ_Booster_StrainA, OmcZ_Booster_StrainB
  quorum_sensing            (2): Luxl_LuxR, LuxS
  kill_switch               (2): KillSwitch_StrainA, KillSwitch_StrainB
  metabolite_production     (4): BetaCarotene, Isoprenoid, Naringenin, MalonylCoA


## Step 3 — Genetic parts reference

Each circuit is a **transcriptional unit**: `Promoter → RBS → CDS₁ → … → CDSₙ → Terminator`.
Kill-switch circuits are composite (two transcriptional units in tandem).

The table below shows the **actual genetic parts** used to build each of the 20 circuits,
grouped by module. This is a reference view — tuning happens with the module sliders
in Step 7.


In [3]:
# ----------------------------------------------------------------------------
# Library of interchangeable parts.
# This is the "shelf" of available promoters / RBSs / terminators that any
# circuit could in principle use. CDSs are not here because they are
# determined by the biology of each circuit (e.g. lysC, omcZ, mazF) rather
# than being freely interchangeable.
# ----------------------------------------------------------------------------
AVAILABLE_PARTS = {
    'promoter':   ['Ptrc', 'Plux', 'PfapR', 'PnirH', 'PomcZ', 'Pconst', 'Prep', 'PlacI', 'PT7'],
    'rbs':        ['RBS1', 'RBS2', 'RBSstrong', 'RBSweak'],
    'terminator': ['T1', 'T2', 'Term1', 'Term2', 'TrrnB', 'TL3'],
}

# ----------------------------------------------------------------------------
# Structured parts per circuit. The value is always a *list* of transcriptional
# units so that composite circuits (e.g. the kill switches, which have an
# antitoxin cassette + a toxin cassette) can be represented with multiple
# entries. Single-cassette circuits just have a one-element list.
#
# Each unit dict has: promoter, rbs, cds (list of genes in order), terminator.
# ----------------------------------------------------------------------------
CIRCUIT_PARTS = {
    # Cross-feeding — each strain's amino-acid biosynthesis cassette
    'Ecoli_K':   [{'promoter':'Ptrc','rbs':'RBS2','cds':['lysC','argD','lysA'],'terminator':'T2'}],
    'Ecoli_L':   [{'promoter':'Ptrc','rbs':'RBS1','cds':['leuB','leuC','leuD'],'terminator':'T1'}],
    'Ecoli_Trp': [{'promoter':'Ptrc','rbs':'RBS1','cds':['trpB','trpA'],       'terminator':'T1'}],
    'Ecoli_His': [{'promoter':'Ptrc','rbs':'RBS1','cds':['hisC','hisD'],       'terminator':'T1'}],
    # Nitrogen cycle — nitrification (amoA/hao, nxrA) + denitrification (narG/nirS, norB/nosZ)
    'Nitrosomonas':             [{'promoter':'Ptrc','rbs':'RBS1','cds':['amoA','hao'], 'terminator':'T1'}],
    'Nitrobacter':              [{'promoter':'Ptrc','rbs':'RBS2','cds':['nxrA'],       'terminator':'T2'}],
    'Pseudomonas_stutzeri':     [{'promoter':'Ptrc','rbs':'RBS1','cds':['narG','nirS'],'terminator':'T1'}],
    'Paracoccus_denitrificans': [{'promoter':'Ptrc','rbs':'RBS2','cds':['norB','nosZ'],'terminator':'T2'}],
    # Electrogenesis — Geobacter chassis + OmcZ over-expression boosters
    'StrainA_Gsulfurreducens':  [{'promoter':'Ptrc', 'rbs':'RBS1','cds':['acs'],  'terminator':'Term1'}],
    'StrainB_Gmetalireducens':  [{'promoter':'PnirH','rbs':'RBS2','cds':['omcZ'], 'terminator':'Term2'}],
    'OmcZ_Booster_StrainA':     [{'promoter':'PomcZ','rbs':'RBS1','cds':['omcZ'], 'terminator':'Term1'}],
    'OmcZ_Booster_StrainB':     [{'promoter':'PomcZ','rbs':'RBS2','cds':['omcZ'], 'terminator':'Term2'}],
    # Quorum sensing — AHL system + LuxS for AI-2
    'Luxl_LuxR': [{'promoter':'Plux','rbs':'RBS1','cds':['luxI','luxR'],'terminator':'T1'}],
    'LuxS':      [{'promoter':'Ptrc','rbs':'RBS1','cds':['luxS'],       'terminator':'T1'}],
    # Kill switches — composite circuits: antitoxin cassette + toxin cassette
    # (separate transcriptional units under different promoters)
    'KillSwitch_StrainA': [
        {'promoter':'Pconst','rbs':'RBS1','cds':['mazE'],'terminator':'Term1'},  # antitoxin (constitutive)
        {'promoter':'Prep',  'rbs':'RBS1','cds':['mazF'],'terminator':'Term1'},  # toxin (KillSignal-responsive)
    ],
    'KillSwitch_StrainB': [
        {'promoter':'Pconst','rbs':'RBS2','cds':['mazE'],'terminator':'Term2'},
        {'promoter':'Prep',  'rbs':'RBS2','cds':['mazF'],'terminator':'Term2'},
    ],
    # Metabolite production — pathway cassettes
    'BetaCarotene': [{'promoter':'Ptrc', 'rbs':'RBS1','cds':['crtE','crtB','crtI'],    'terminator':'T1'}],
    'Isoprenoid':   [{'promoter':'Ptrc', 'rbs':'RBS1','cds':['atoB','HMGS','HMGR'],    'terminator':'T1'}],
    'Naringenin':   [{'promoter':'Ptrc', 'rbs':'RBS1','cds':['TAL','4CL','CHS','CHI'], 'terminator':'T1'}],
    'MalonylCoA':   [{'promoter':'PfapR','rbs':'RBS1','cds':['fapR','fabH'],           'terminator':'T1'}],
}

# Make sure the parts dict and the circuit dict stay in sync — fail fast otherwise.
assert set(CIRCUIT_PARTS) == set(CIRCUITS), "Parts must be defined for every circuit"


def unit_to_spec(u):
    """Render one transcriptional unit as a GOLDBAR `then` chain.

    e.g. {'promoter':'Ptrc','rbs':'RBS1','cds':['amoA','hao'],'terminator':'T1'}
       -> 'Ptrc then RBS1 then amoA then hao then T1'
    """
    return ' then '.join([u['promoter'], u['rbs'], *u['cds'], u['terminator']])

def circuit_to_spec(cname):
    """Render a whole circuit (possibly multi-unit) as a GOLDBAR string.

    Single-unit circuits are returned as-is. Composite circuits get each
    unit wrapped in parentheses and joined with `then`, matching the
    GOLDBAR convention for sequential composition of cassettes.
    """
    specs = [unit_to_spec(u) for u in CIRCUIT_PARTS[cname]]
    return specs[0] if len(specs) == 1 else ' then '.join(f'({s})' for s in specs)

# Keep CIRCUITS[*]['spec'] as the single source of truth derived from
# CIRCUIT_PARTS, so editing parts in one place updates everywhere.
for cname in CIRCUITS:
    CIRCUITS[cname]['spec'] = circuit_to_spec(cname)

# Quick visual confirmation that single-unit and composite renderings work.
print('Parts defined for', len(CIRCUIT_PARTS), 'circuits.')
print('Sample:', 'Ecoli_K ->', circuit_to_spec('Ecoli_K'))
print('Sample:', 'KillSwitch_StrainA ->', circuit_to_spec('KillSwitch_StrainA'))


Parts defined for 20 circuits.
Sample: Ecoli_K -> Ptrc then RBS2 then lysC then argD then lysA then T2
Sample: KillSwitch_StrainA -> (Pconst then RBS1 then mazE then Term1) then (Prep then RBS1 then mazF then Term1)


In [4]:
# ----------------------------------------------------------------------------
# Read-only "genetic parts table" widget. Despite the name `parts_editor`,
# this version is purely a *reference view* — no dropdowns, no editing.
# Tuning happens later via the module sliders in Step 7.
# ----------------------------------------------------------------------------
MODULE_ORDER = ['cross_feeding','nitrogen_cycle','electrogenesis',
                'quorum_sensing','kill_switch','metabolite_production']

# Display labels with numbered emoji prefixes (renders nicely in Jupyter HTML).
MODULE_LABEL = {
    'cross_feeding':         '1️⃣  Cross-feeding',
    'nitrogen_cycle':        '2️⃣  Nitrogen cycle',
    'electrogenesis':        '3️⃣  Electrogenesis',
    'quorum_sensing':        '4️⃣  Quorum sensing',
    'kill_switch':           '5️⃣  Kill switch',
    'metabolite_production': '6️⃣  Metabolite production',
}

# --- Render as static read-only rows (no dropdowns — reference only) ---
def _static_row(cname, unit_idx, is_first):
    """Build one row of the parts table for a single transcriptional unit.

    For composite circuits (kill switches), the first unit shows the circuit
    name; subsequent units show an indented "↳ unit 2" marker so it's clear
    they belong to the same circuit.
    """
    u = CIRCUIT_PARTS[cname][unit_idx]
    # First unit gets the bold circuit name; later units get a faded continuation marker.
    name_html = (f'<b>{cname}</b>' if is_first
                 else '<i style="color:#888">  &nbsp;↳ unit 2</i>')
    # Shared style for the four data cells (promoter / RBS / CDS / terminator).
    cell_style = ('background:#e2e8f0;color:#0f172a;padding:5px 10px;'
                  'border-radius:4px;font-family:monospace;')
    return widgets.HTML(
        f'<div style="display:flex;gap:6px;align-items:center;padding:2px 0;">'
        f'  <div style="width:220px">{name_html}</div>'
        f'  <div style="{cell_style}width:90px;">{u["promoter"]}</div>'
        f'  <div style="{cell_style}width:90px;">{u["rbs"]}</div>'
        f'  <div style="{cell_style}flex:1;min-width:240px;">{", ".join(u["cds"])}</div>'
        f'  <div style="{cell_style}width:90px;">{u["terminator"]}</div>'
        f'</div>'
    )


def build_parts_editor():
    """Assemble the full table: header row, then per-module sections of circuits."""
    out = []
    # --- Column headers (Circuit | Promoter | RBS | CDS list | Terminator) ---
    hdr_style = 'color:#334155;font-weight:bold;'
    out.append(widgets.HTML(
        f'<div style="display:flex;gap:6px;padding:4px 0;border-bottom:1px solid #cbd5e1;">'
        f'  <div style="width:220px;{hdr_style}">Circuit</div>'
        f'  <div style="width:90px;{hdr_style}">Promoter</div>'
        f'  <div style="width:90px;{hdr_style}">RBS</div>'
        f'  <div style="flex:1;min-width:240px;{hdr_style}">CDS list</div>'
        f'  <div style="width:90px;{hdr_style}">Terminator</div>'
        f'</div>'
    ))

    # --- One section per module, with a dark banner heading ---
    for mod in MODULE_ORDER:
        circuits_in_mod = [n for n, c in CIRCUITS.items() if c['module'] == mod]
        if not circuits_in_mod: continue  # skip empty modules (none in current library, but defensive)
        # Module banner row.
        out.append(widgets.HTML(
            f'<div style="background:#1e293b;color:white;padding:6px 10px;'
            f'margin-top:10px;border-radius:4px;"><b>{MODULE_LABEL[mod]}</b> '
            f'<span style="color:#94a3b8;">({len(circuits_in_mod)} circuits)</span></div>'
        ))
        # One row per transcriptional unit in each circuit.
        for cname in circuits_in_mod:
            for i in range(len(CIRCUIT_PARTS[cname])):
                out.append(_static_row(cname, i, is_first=(i == 0)))
            # GOLDBAR spec line under each circuit (computed in Step 3).
            out.append(widgets.HTML(
                f'<div style="padding-left:220px;color:#64748b;font-size:11px;'
                f'margin-bottom:6px;">→ GOLDBAR: '
                f'<code style="color:#0ea5e9;background:#f1f5f9;'
                f'padding:3px 6px;border-radius:4px;">{CIRCUITS[cname]["spec"]}</code></div>'
            ))

    return widgets.VBox(out)

# Build and display the table immediately when this cell runs.
parts_editor = build_parts_editor()
display(widgets.HTML(
    '<h3 style="margin:0 0 6px 0;">🧬 Genetic parts table '
    '(20 circuits · 22 transcriptional units)</h3>'
    '<p style="color:#64748b;margin:0 0 10px 0;">'
    'Reference only — this shows the actual GOLDBAR parts for each circuit. '
    'Tuning happens with the module sliders further down.</p>'
))
display(parts_editor)


HTML(value='<h3 style="margin:0 0 6px 0;">🧬 Genetic parts table (20 circuits · 22 transcriptional units)</h3><…

## Step 4 — The ODE model

Mirror of the SBML `whole_ecosystem_20circuits.xml` model: 33 species,
49 reactions. Each slider multiplies the rate constants of its module by a
value in `[0, 1]`, so moving a slider to `0` effectively switches that
module off, and `1` gives the native dynamics.


In [5]:
# ============================================================================
# Species (33 state variables). The order here defines indices into the
# state vector y[]; IDX[name] -> integer index lets us read/write specific
# species by name in the rhs() function below.
# Mirror of the species set in whole_ecosystem_20circuits.xml.
# ============================================================================
SPECIES = [
    # Cross-feeding: 4 strains + 4 amino-acid pools
    'Ecoli_K','Ecoli_L','Ecoli_Trp','Ecoli_His',
    'Lys','Leu','Trp','His',
    # Electrogenesis: 2 Geobacter strains + Acs/OmcZ enzymes + electrical Current
    'Gsulf','Gmet','Acs','OmcZ_A','OmcZ_B','Current',
    # Kill switch: MazE/MazF antitoxin/toxin pairs per strain + KillSignal trigger
    'MazE_A','MazF_A','MazE_B','MazF_B','KillSignal',
    # Quorum sensing: AHL signal, AHL-bound LuxR, AI-2 universal signal
    'AHL','LuxR_AHL','AI2',
    # Nitrogen cycle: full cascade NH4 -> NO2 -> NO3 -> NO -> N2O -> N2
    'NH4','NO2','NO3','NO','N2O','N2',
    # Metabolite production: AcetylCoA branches into MalonylCoA->Naringenin
    # and Isoprenoid->BetaCarotene
    'AcetylCoA','MalonylCoA','Naringenin','Isoprenoid','BetaCarotene',
]
# Lookup: species name -> position in y[]. Lets the ODEs use IDX['Gsulf'] instead of 8.
IDX = {s:i for i,s in enumerate(SPECIES)}

# ----------------------------------------------------------------------------
# Initial conditions. Most species start at 0 (build up from reactions);
# only the seeded biomass and the upstream substrates have non-zero starts.
# ----------------------------------------------------------------------------
Y0 = np.zeros(len(SPECIES))
# Seed each E. coli strain at 1.0 cell-units
Y0[IDX['Ecoli_K']]   = 1.0
Y0[IDX['Ecoli_L']]   = 1.0
Y0[IDX['Ecoli_Trp']] = 1.0
Y0[IDX['Ecoli_His']] = 1.0
# Seed each Geobacter strain at 1.0
Y0[IDX['Gsulf']]     = 1.0
Y0[IDX['Gmet']]      = 1.0
# Pre-load antitoxin pools so cells survive until KillSignal builds up
Y0[IDX['MazE_A']]    = 5.0
Y0[IDX['MazE_B']]    = 5.0
# Upstream substrates for the nitrogen cycle and metabolite pathways
Y0[IDX['NH4']]       = 10.0
Y0[IDX['AcetylCoA']] = 5.0

# ----------------------------------------------------------------------------
# Base rate constants (numerically identical to the SBML model).
# Each entry is (k_base, module): the slider for that module multiplies k_base
# in [0, 1] — slider=0 turns the reaction off, slider=1 gives native rate.
# This module-tagging is what lets a single slider control a coherent group
# of reactions without having to wire each one up individually.
# ----------------------------------------------------------------------------
RATES = {
    # cross-feeding: amino-acid production, growth-on-AA, and death
    'k_K_lys':    (0.15, 'cross_feeding'),
    'k_L_leu':    (0.15, 'cross_feeding'),
    'k_Trp_trp':  (0.15, 'cross_feeding'),
    'k_His_his':  (0.15, 'cross_feeding'),
    'k_K_grow':   (0.04, 'cross_feeding'),
    'k_L_grow':   (0.04, 'cross_feeding'),
    'k_Trp_grow': (0.04, 'cross_feeding'),
    'k_His_grow': (0.04, 'cross_feeding'),
    'k_K_death':   (0.05, 'cross_feeding'),
    'k_L_death':   (0.05, 'cross_feeding'),
    'k_Trp_death': (0.05, 'cross_feeding'),
    'k_His_death': (0.05, 'cross_feeding'),
    # electrogenesis: Geobacter dynamics + OmcZ expression + current generation
    'k_Gsulf_grow':    (0.03, 'electrogenesis'),
    'k_Gmet_grow':     (0.03, 'electrogenesis'),
    'k_Gsulf_death':   (0.10, 'electrogenesis'),
    'k_Gmet_death':    (0.10, 'electrogenesis'),
    'k_acs':           (0.10, 'electrogenesis'),
    'k_omcZ_B_basal':  (0.05, 'electrogenesis'),
    'k_omcZ_boost_A':  (0.20, 'electrogenesis'),
    'k_omcZ_boost_B':  (0.20, 'electrogenesis'),
    'k_current':       (0.02, 'electrogenesis'),
    'k_NO_induces_omcZ': (0.08, 'electrogenesis'),  # cross-talk: NO from N-cycle induces OmcZ in StrainB
    # kill switch: MazE/MazF toxin-antitoxin dynamics
    'k_mazE_A':   (0.10, 'kill_switch'),  # MazE production
    'k_mazE_B':   (0.10, 'kill_switch'),
    'k_mazF_A':   (0.15, 'kill_switch'),  # MazF production (gated by KillSignal)
    'k_mazF_B':   (0.15, 'kill_switch'),
    'k_antitox':  (0.50, 'kill_switch'),  # MazE neutralises MazF (consumes both)
    'k_tox_kill': (0.25, 'kill_switch'),  # free MazF kills its host strain
    'k_mazE_deg': (0.05, 'kill_switch'),  # spontaneous antitoxin decay
    # quorum sensing: AHL/AI-2 dynamics + KillSignal coupling
    'k_luxI':        (0.05, 'quorum_sensing'),
    'k_luxS':        (0.05, 'quorum_sensing'),
    'k_luxR_act':    (0.10, 'quorum_sensing'),  # AHL binds LuxR
    'k_killsig':     (0.05, 'quorum_sensing'),  # LuxR-AHL drives KillSignal production
    'k_AHL_deg':     (0.02, 'quorum_sensing'),
    'k_AI2_deg':     (0.02, 'quorum_sensing'),
    'k_killsig_deg': (0.05, 'quorum_sensing'),
    # nitrogen cycle: nitrification (k_nitroso/k_nitroba) + denitrification (k_pseud_*, k_parac_*)
    'k_nitrosomonas': (0.15, 'nitrogen_cycle'),
    'k_nitrobacter':  (0.12, 'nitrogen_cycle'),
    'k_pseud_narG':   (0.10, 'nitrogen_cycle'),
    'k_pseud_nirS':   (0.10, 'nitrogen_cycle'),
    'k_parac_norB':   (0.10, 'nitrogen_cycle'),
    'k_parac_nosZ':   (0.10, 'nitrogen_cycle'),
    # metabolite production: AcetylCoA -> MalonylCoA -> Naringenin
    #                        AcetylCoA -> Isoprenoid  -> BetaCarotene
    'k_malCoA':     (0.08, 'metabolite_production'),
    'k_naringenin': (0.06, 'metabolite_production'),
    'k_isoprenoid': (0.08, 'metabolite_production'),
    'k_betacarot':  (0.06, 'metabolite_production'),
}

def effective_rates(weights):
    """Return a flat dict of {rate_name: scaled_k}.

    Each base rate constant is multiplied by its module's slider weight.
    weights={m: 1 for all m} reproduces the native SBML rates;
    weights[m]=0 effectively switches module m off without removing reactions.
    """
    return {k: base * weights[mod] for k,(base, mod) in RATES.items()}

def rhs(t, y, p):
    """Right-hand side of the 33-equation ODE system.

    Parameters
    ----------
    t : float
        Current time (unused — system is autonomous, kept for solve_ivp signature).
    y : ndarray, shape (33,)
        Current state (one slot per species in SPECIES order).
    p : dict
        Effective rate constants from effective_rates(weights).

    Returns
    -------
    dy : ndarray, shape (33,)
        Time derivatives for every species.
    """
    # Unpack y into named locals — keeps the rate expressions readable.
    Ecoli_K, Ecoli_L, Ecoli_Trp, Ecoli_His = y[0], y[1], y[2], y[3]
    Lys, Leu, Trp, His = y[4], y[5], y[6], y[7]
    Gsulf, Gmet, Acs, OmcZ_A, OmcZ_B, Current = y[8], y[9], y[10], y[11], y[12], y[13]
    MazE_A, MazF_A, MazE_B, MazF_B, KillSignal = y[14], y[15], y[16], y[17], y[18]
    AHL, LuxR_AHL, AI2 = y[19], y[20], y[21]
    NH4, NO2, NO3, NO_, N2O, N2 = y[22], y[23], y[24], y[25], y[26], y[27]
    AcetylCoA, MalonylCoA, Naringenin, Isoprenoid, BetaCarotene = y[28], y[29], y[30], y[31], y[32]

    # Clamp biomass to non-negative *for rate calculations only* (the solver
    # may dip slightly below zero between steps; this avoids negative-feedback
    # blow-ups without masking real dynamics in the returned derivatives).
    Ecoli_K = max(Ecoli_K, 0); Ecoli_L = max(Ecoli_L, 0)
    Ecoli_Trp = max(Ecoli_Trp, 0); Ecoli_His = max(Ecoli_His, 0)
    Gsulf = max(Gsulf, 0); Gmet = max(Gmet, 0)

    # --- Cross-feeding reactions ---
    # Each strain catalytically produces "its" amino acid (rate ∝ strain only)…
    v_K_lys   = p['k_K_lys']   * Ecoli_K
    v_L_leu   = p['k_L_leu']   * Ecoli_L
    v_Trp_prod= p['k_Trp_trp'] * Ecoli_Trp
    v_His_prod= p['k_His_his'] * Ecoli_His
    # …and grows on the AA it cannot make itself (rate ∝ strain * substrate AA).
    v_K_grow   = p['k_K_grow']   * Ecoli_K   * max(Leu, 0)
    v_L_grow   = p['k_L_grow']   * Ecoli_L   * max(Lys, 0)
    v_Trp_grow = p['k_Trp_grow'] * Ecoli_Trp * max(His, 0)
    v_His_grow = p['k_His_grow'] * Ecoli_His * max(Trp, 0)
    # First-order death of each strain (dilution / lysis).
    v_K_death   = p['k_K_death']   * Ecoli_K
    v_L_death   = p['k_L_death']   * Ecoli_L
    v_Trp_death = p['k_Trp_death'] * Ecoli_Trp
    v_His_death = p['k_His_death'] * Ecoli_His

    # --- Electrogenesis ---
    # Geobacter growth/death (independent of substrate in this simplified model)
    v_Gsulf_grow  = p['k_Gsulf_grow']  * Gsulf
    v_Gmet_grow   = p['k_Gmet_grow']   * Gmet
    v_Gsulf_death = p['k_Gsulf_death'] * Gsulf
    v_Gmet_death  = p['k_Gmet_death']  * Gmet
    v_acs         = p['k_acs']         * Gsulf            # acs expression in StrainA
    v_omcZ_B_base = p['k_omcZ_B_basal']* Gmet             # leaky baseline OmcZ in StrainB
    v_omcZ_B_ind  = p['k_NO_induces_omcZ'] * Gmet * max(NO_, 0)  # PnirH induction by NO (cross-talk with N-cycle)
    v_omcZ_bst_A  = p['k_omcZ_boost_A']* Gsulf            # extra PomcZ->omcZ booster cassette
    v_omcZ_bst_B  = p['k_omcZ_boost_B']* Gmet
    # Current generation: needs both strains; OmcZ levels boost it multiplicatively
    # (the +1 keeps current non-zero even when OmcZ is still 0).
    v_current     = p['k_current'] * Gsulf * Gmet * (1 + max(OmcZ_A,0)) * (1 + max(OmcZ_B,0))

    # --- Kill switch (MazE = antitoxin, MazF = toxin) ---
    v_mazE_A   = p['k_mazE_A']   * Gsulf                  # constitutive antitoxin
    v_mazF_A   = p['k_mazF_A']   * max(KillSignal,0) * Gsulf  # toxin only made when KillSignal is on
    v_antitox_A= p['k_antitox']  * max(MazE_A,0) * max(MazF_A,0)  # antitoxin neutralises toxin (consumes both)
    v_kill_A   = p['k_tox_kill'] * max(MazF_A,0) * Gsulf  # free toxin -> strain death
    v_mazE_A_dg= p['k_mazE_deg'] * max(MazE_A,0)          # antitoxin spontaneous decay
    v_mazE_B   = p['k_mazE_B']   * Gmet
    v_mazF_B   = p['k_mazF_B']   * max(KillSignal,0) * Gmet
    v_antitox_B= p['k_antitox']  * max(MazE_B,0) * max(MazF_B,0)
    v_kill_B   = p['k_tox_kill'] * max(MazF_B,0) * Gmet
    v_mazE_B_dg= p['k_mazE_deg'] * max(MazE_B,0)

    # --- Quorum sensing ---
    v_AHL     = p['k_luxI']     * Gsulf                   # LuxI in StrainA secretes AHL
    v_luxR    = p['k_luxR_act'] * max(AHL,0)              # AHL binds LuxR (consumed -> LuxR_AHL)
    v_killsig = p['k_killsig']  * max(LuxR_AHL,0)         # active LuxR drives KillSignal expression
    v_AHL_dg  = p['k_AHL_deg']  * max(AHL,0)
    v_ksig_dg = p['k_killsig_deg'] * max(KillSignal,0)
    v_AI2     = p['k_luxS']     * Gmet                    # universal signal from StrainB
    v_AI2_dg  = p['k_AI2_deg']  * max(AI2,0)

    # --- Nitrogen cycle (linear cascade: NH4 -> NO2 -> NO3 -> NO -> N2O -> N2) ---
    v_nitroso = p['k_nitrosomonas'] * max(NH4,0)
    v_nitroba = p['k_nitrobacter']  * max(NO2,0)
    v_narG    = p['k_pseud_narG']   * max(NO3,0)          # P. stutzeri narG: NO3 -> NO2
    v_nirS    = p['k_pseud_nirS']   * max(NO2,0)          #               nirS: NO2 -> NO
    v_norB    = p['k_parac_norB']   * max(NO_,0)          # P. denitrificans norB: NO -> N2O
    v_nosZ    = p['k_parac_nosZ']   * max(N2O,0)          #                   nosZ: N2O -> N2

    # --- Metabolite production (two parallel branches off AcetylCoA) ---
    v_malCoA    = p['k_malCoA']     * max(AcetylCoA,0)    # AcetylCoA -> MalonylCoA
    v_naringen  = p['k_naringenin'] * max(MalonylCoA,0)   # MalonylCoA -> Naringenin
    v_isoprenoid= p['k_isoprenoid'] * max(AcetylCoA,0)    # AcetylCoA -> Isoprenoid
    v_betacarot = p['k_betacarot']  * max(Isoprenoid,0)   # Isoprenoid -> BetaCarotene

    # ------------------------------------------------------------------------
    # Assemble derivatives. For each species, sum of incoming reactions minus
    # outgoing. Comments highlight any non-obvious coupling.
    # ------------------------------------------------------------------------
    dy = np.zeros(len(SPECIES))

    # Cross-feeding biomass: AA-producing reactions are catalytic (don't consume
    # the strain), so only growth and death change strain levels.
    dy[IDX['Ecoli_K']]   = v_K_grow   - v_K_death
    dy[IDX['Ecoli_L']]   = v_L_grow   - v_L_death
    dy[IDX['Ecoli_Trp']] = v_Trp_grow - v_Trp_death
    dy[IDX['Ecoli_His']] = v_His_grow - v_His_death
    # AA pools: produced by their auxotroph-complement strain, consumed by the
    # strain that needs them for growth (cross-feeding partnership).
    dy[IDX['Lys']] = v_K_lys    - v_L_grow
    dy[IDX['Leu']] = v_L_leu    - v_K_grow
    dy[IDX['Trp']] = v_Trp_prod - v_His_grow
    dy[IDX['His']] = v_His_prod - v_Trp_grow

    # Electrogenesis: kill-switch deaths feed back into Geobacter biomass here.
    dy[IDX['Gsulf']]   = v_Gsulf_grow - v_Gsulf_death - v_kill_A
    dy[IDX['Gmet']]    = v_Gmet_grow  - v_Gmet_death  - v_kill_B
    dy[IDX['Acs']]     = v_acs                          # accumulated Acs (no degradation modelled)
    dy[IDX['OmcZ_A']]  = v_omcZ_bst_A
    dy[IDX['OmcZ_B']]  = v_omcZ_B_base + v_omcZ_B_ind + v_omcZ_bst_B
    dy[IDX['Current']] = v_current                      # cumulative current output

    # Kill switch: antitoxin balance = production - neutralisation - decay.
    # Toxin balance = production - neutralisation (the kill reaction is
    # bookkept via the strain's death term, not a toxin sink).
    dy[IDX['MazE_A']] = v_mazE_A - v_antitox_A - v_mazE_A_dg
    dy[IDX['MazF_A']] = v_mazF_A - v_antitox_A
    dy[IDX['MazE_B']] = v_mazE_B - v_antitox_B - v_mazE_B_dg
    dy[IDX['MazF_B']] = v_mazF_B - v_antitox_B
    dy[IDX['KillSignal']] = v_killsig - v_ksig_dg

    # Quorum sensing: AHL is consumed when binding LuxR (-> LuxR_AHL pool grows).
    dy[IDX['AHL']]      = v_AHL - v_luxR - v_AHL_dg
    dy[IDX['LuxR_AHL']] = v_luxR
    dy[IDX['AI2']]      = v_AI2 - v_AI2_dg

    # Nitrogen cycle: each species is produced by the upstream step and
    # consumed by the downstream step. NO2 is special — it has TWO sources
    # (nitrification by Nitrosomonas + denitrification narG step) and two sinks.
    dy[IDX['NH4']] = -v_nitroso
    dy[IDX['NO2']] =  v_nitroso - v_nitroba + v_narG - v_nirS
    dy[IDX['NO3']] =  v_nitroba - v_narG
    dy[IDX['NO']]  =  v_nirS    - v_norB
    dy[IDX['N2O']] =  v_norB    - v_nosZ
    dy[IDX['N2']]  =  v_nosZ                            # terminal sink (no further reactions)

    # Metabolite production: AcetylCoA feeds both branches simultaneously.
    dy[IDX['AcetylCoA']]    = -v_malCoA - v_isoprenoid
    dy[IDX['MalonylCoA']]   =  v_malCoA - v_naringen
    dy[IDX['Naringenin']]   =  v_naringen
    dy[IDX['Isoprenoid']]   =  v_isoprenoid - v_betacarot
    dy[IDX['BetaCarotene']] =  v_betacarot              # terminal product

    return dy


def simulate(weights, t_end=50.0, n_points=101):
    """Integrate the ODE system from t=0 to t=t_end with given module weights.

    Uses LSODA — robust on stiff systems (the kill-switch dynamics can get
    stiff once KillSignal builds up, and the nitrogen cascade has very
    different timescales for adjacent steps).

    Returns the full scipy `OdeSolution` object (sol.t, sol.y, sol.success...).
    """
    p = effective_rates(weights)
    t_eval = np.linspace(0, t_end, n_points)   # uniform grid for plotting
    sol = solve_ivp(rhs, (0, t_end), Y0, args=(p,),
                    method='LSODA', rtol=1e-6, atol=1e-9, t_eval=t_eval)
    return sol


# Smoke test: run once with all sliders at 1.0 and report the final values
# of two terminal species. If this prints sensible numbers, the model is wired.
test_weights = {m: 1.0 for m in MODULES}
sol = simulate(test_weights)
print(f"Test sim: success={sol.success}, final Current={sol.y[IDX['Current']][-1]:.3f}, "
      f"final N2={sol.y[IDX['N2']][-1]:.3f}")


Test sim: success=True, final Current=0.792, final N2=6.195

## Step 5 — Circuit selection from slider weights

In [6]:
def score_circuits(weights):
    """Dot product of each circuit's score vector with slider weights.

    Since each circuit's score vector is one-hot on its module, this reduces
    to: score = weights[module_of_that_circuit]. Implemented as a general
    dot product anyway so it stays correct if multi-module circuits are ever
    added in the future.
    """
    return {name: sum(weights[m] * info['scores'][m] for m in MODULES)
            for name, info in CIRCUITS.items()}

def select_consortium(weights, threshold=0.25):
    """Pick the active consortium given current slider weights.

    Steps:
      1. Compute every circuit's score.
      2. Keep circuits at or above `threshold`.
      3. Apply *dependency closure*: if a selected circuit `requires` other
         circuits (e.g. a kill switch needs its host strain), pull those in
         too. Iterate until no new circuits are added (fixed point).

    Returns
    -------
    (selected_sorted, scores) : (list[str], dict[str, float])
        Selected circuit names sorted by descending score, plus the full
        score map (used to render the fitness bar chart).
    """
    scores = score_circuits(weights)
    selected = {n for n,s in scores.items() if s >= threshold}

    # Dependency closure: keep adding required circuits until the set stops growing.
    # Using a while-loop with a `changed` flag handles transitive requirements
    # correctly (A requires B, B requires C — both B and C get pulled in).
    changed = True
    while changed:
        changed = False
        for n in list(selected):                    # snapshot via list() — we mutate `selected`
            for req in CIRCUITS[n]['requires']:
                if req not in selected:
                    selected.add(req); changed = True

    # Sort by score so the highest-scoring circuits show first in the table.
    return sorted(selected, key=lambda n: -scores[n]), scores


## Step 6 — The live renderer

Draws one big plot (iBioSim-style) with a species picker, plus the
consortium table and per-circuit fitness bars. Everything updates in under
a second when a slider moves.


In [7]:
# ----------------------------------------------------------------------------
# Per-module colors used in the fitness bar chart. Distinct hues so the
# six modules are easily told apart even at low chart resolution.
# ----------------------------------------------------------------------------
MODULE_COLORS = {
    'cross_feeding':         '#8FD14F',
    'nitrogen_cycle':        '#A855F7',
    'electrogenesis':        '#00B4B4',
    'quorum_sensing':        '#3B82F6',
    'kill_switch':           '#F59E0B',
    'metabolite_production': '#EF4444',
}

# ----------------------------------------------------------------------------
# Preset species groupings — each is a one-click "view" the user can pick to
# focus the time-series plot on a particular subsystem instead of all 33.
# ----------------------------------------------------------------------------
SPECIES_PRESETS = {
    'iBioSim default (Electrogenesis + Kill switch)':
        ['AHL','Current','Gmet','Gsulf','KillSignal','MazE_A','OmcZ_A','OmcZ_B'],
    'Cross-feeding':
        ['Ecoli_K','Ecoli_L','Ecoli_Trp','Ecoli_His','Lys','Leu','Trp','His'],
    'Nitrogen cycle':
        ['NH4','NO2','NO3','NO','N2O','N2'],
    'Metabolite production':
        ['AcetylCoA','MalonylCoA','Naringenin','Isoprenoid','BetaCarotene'],
    'Quorum sensing':
        ['AHL','LuxR_AHL','AI2','KillSignal','MazF_A','MazF_B'],
    'All 33 species (full ecosystem)':
        list(SPECIES),
}

# ----------------------------------------------------------------------------
# Color palette for the time-series plot — one stable color per species,
# assigned by index so a given species always plots in the same color across
# different runs / preset switches.
# ----------------------------------------------------------------------------
_PALETTE = [
    '#E74C3C','#3498DB','#2ECC71','#F39C12','#9B59B6','#1ABC9C','#E67E22','#34495E',
    '#16A085','#D35400','#8E44AD','#27AE60','#C0392B','#2980B9','#F1C40F','#7F8C8D',
    '#E91E63','#00BCD4','#CDDC39','#795548','#607D8B','#FF5722','#009688','#FFC107',
    '#673AB7','#FF9800','#4CAF50','#F44336','#2196F3','#9C27B0','#3F51B5','#00ACC1',
    '#8BC34A',
]
SPECIES_COLOR = {s: _PALETTE[i % len(_PALETTE)] for i,s in enumerate(SPECIES)}


def render(weights, species_to_plot, threshold=0.25, t_end=50.0, log_y=False):
    """Top-level renderer: consortium table + simulation plot + fitness bars.

    Called once at startup and again every time a slider changes (see Step 7).
    Output order is intentional — the table shows what *will* be simulated,
    then the plot shows the dynamics, then the bar chart explains *why* each
    circuit is in (or out of) the consortium.
    """
    # --- 1. Consortium table (Markdown) -------------------------------------
    selected, scores = select_consortium(weights, threshold)
    if selected:
        # Build a Markdown table sorted by score (selected list is already sorted).
        md = [f"### 🧬 Selected consortium — {len(selected)} of 20 circuits\n",
              "| Score | Circuit | Module | Organism |",
              "|---|---|---|---|"]
        for n in selected:
            c = CIRCUITS[n]
            md.append(f"| **{scores[n]:.2f}** | `{n}` | `{c['module']}` | *{c['organism']}* |")
        display(Markdown('\n'.join(md)))
    else:
        # No circuit cleared the threshold — guide the user to raise sliders.
        display(Markdown("### ⚠️ No circuits cleared the threshold — raise some sliders."))

    # --- 2. Run simulation --------------------------------------------------
    sol = simulate(weights, t_end=t_end)
    if not sol.success:
        # Bail out gracefully if the integrator fails (e.g. extreme parameter combo).
        print("⚠️ Integration failed:", sol.message); return

    # --- 3. Time-series plot (iBioSim TSD-style single panel) ---------------
    if species_to_plot:
        fig, ax = plt.subplots(figsize=(14, 7.5))
        # One line per species, with markers at every 5th time point so the
        # eye can pick individual lines apart even when they overlap.
        for s in species_to_plot:
            ax.plot(sol.t, sol.y[IDX[s]],
                    label=s, color=SPECIES_COLOR[s],
                    linewidth=1.8, marker='.', markersize=3, markevery=5)
        ax.set_title(f"sim1 simulation results  (t = 0 ... {t_end:.0f})",
                     fontsize=14, fontweight='bold')
        ax.set_xlabel('time (minutes)', fontsize=11)
        ax.set_ylabel('amount (molecules / cells)', fontsize=11)
        ax.set_xlim(0, t_end)
        # Fixed y-axis range matches the iBioSim default — keeps the plot
        # comparable across different slider settings (no auto-zoom artifacts).
        ax.set_ylim(0, 5.5)
        # Legend in up to 4 columns to handle the all-33-species preset gracefully.
        ax.legend(loc='best', ncol=min(4, len(species_to_plot)),
                  fontsize=9, framealpha=0.9)
        ax.grid(True, alpha=0.3)
        if log_y:
            ax.set_yscale('log')
        # Hide top/right spines for a cleaner look.
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.show()
    else:
        display(Markdown("*(no species selected for plotting)*"))

    # --- 4. Per-circuit fitness bar chart -----------------------------------
    # Selected circuits get their module's color; un-selected are washed-out grey.
    # Threshold is drawn as a red dashed vertical line for instant readability.
    fig, ax = plt.subplots(figsize=(11, 6))
    names = list(CIRCUITS.keys())
    vals  = [scores[n] for n in names]
    colors= [MODULE_COLORS[CIRCUITS[n]['module']] if n in selected else '#D0D9E0' for n in names]
    ax.barh(names, vals, color=colors, edgecolor='#0B1F2A', linewidth=0.5)
    ax.axvline(threshold, color='#EF4444', ls='--', lw=1.5, label=f'threshold = {threshold}')
    ax.invert_yaxis()  # first circuit on top — matches dict iteration order
    ax.set_xlabel('Fitness score'); ax.set_xlim(0, 1.05)
    ax.legend(loc='lower right', fontsize=10)
    ax.set_title('Per-circuit fitness (coloured = selected)', fontsize=12)
    for spine in ['top','right']:
        ax.spines[spine].set_visible(False)
    plt.tight_layout()
    plt.show()

print("Renderer ready.")


Renderer ready.


---
## 🎛️ Tune your consortium

Each slider scales the rate constants for its module between **0 (off)** and
**1 (native strength)**. The circuit table, 2×2 simulation panel, and
fitness bars redraw after every move.

**Preset scenarios to try:**
- **Full ecosystem** → every slider at `1.0`
- **Brewery wastewater MFC** → cross-feeding `1.0`, electrogenesis `1.0`, kill switch `0.6`, everything else low
- **Municipal N-removal** → nitrogen cycle `1.0`, cross-feeding `0.3`, electrogenesis off
- **Bioproduction** → metabolite production `1.0`, cross-feeding `0.5`, quorum sensing `0.5`
- **Maximum containment** → kill switch `1.0`, quorum sensing `1.0` (QS triggers the kill switch)


In [8]:
# ----------------------------------------------------------------------------
# Slider styling — generous label width so long names like
# "Metabolite production" don't get truncated.
# ----------------------------------------------------------------------------
slider_style  = {'description_width': '200px'}
slider_layout = widgets.Layout(width='560px')

def _slider(desc, val=0.7):
    """Make a 0.0–1.0 FloatSlider with the project's standard styling.

    `continuous_update=False` means the callback only fires when the user
    *releases* the slider, not on every pixel of drag — keeps the simulation
    from running needlessly while a user is sweeping a value.
    """
    return widgets.FloatSlider(value=val, min=0, max=1, step=0.05,
                               description=desc, style=slider_style,
                               layout=slider_layout, continuous_update=False)

# --- Module sliders (the only user controls) ---
# Default value 0.7 is chosen so every module starts with non-trivial activity
# but no single one dominates — gives a reasonable baseline scene to tune from.
s_cross   = _slider('Cross-feeding:')
s_nitr    = _slider('Nitrogen cycle:')
s_electro = _slider('Electrogenesis:')
s_quorum  = _slider('Quorum sensing:')
s_kill    = _slider('Kill switch:')
s_prod    = _slider('Metabolite production:')

# --- Hardcoded baseline settings (tweak these to change the default scene) ---
SELECTION_CUTOFF = 0.5    # circuits with score >= 0.5 enter the consortium
SIM_TIME_HORIZON = 100.0  # minutes to integrate
PLOT_SPECIES = ['AHL','Current','Gmet','Gsulf','KillSignal','MazE_A','OmcZ_A','OmcZ_B']  # iBioSim default view

# Output area where render() draws — clear_output(wait=True) on each redraw
# avoids the flicker of removing-then-redrawing.
output_area = widgets.Output()

def on_change(change=None):
    """Slider-change callback.

    Reads every slider's current value into a `weights` dict, clears the
    output area, and calls render() to redo the table / plot / bars.
    """
    weights = {
        'cross_feeding':         s_cross.value,
        'nitrogen_cycle':        s_nitr.value,
        'electrogenesis':        s_electro.value,
        'quorum_sensing':        s_quorum.value,
        'kill_switch':           s_kill.value,
        'metabolite_production': s_prod.value,
    }
    with output_area:
        clear_output(wait=True)
        render(weights, species_to_plot=PLOT_SPECIES,
               threshold=SELECTION_CUTOFF, t_end=SIM_TIME_HORIZON,
               log_y=False)

# Wire every slider to the same callback. `names='value'` means we only
# fire when the slider's value changes (not when, e.g., its layout changes).
for w in [s_cross, s_nitr, s_electro, s_quorum, s_kill, s_prod]:
    w.observe(on_change, names='value')

# Lay out the sliders vertically with a header above them.
controls = widgets.VBox([
    widgets.HTML('<h3 style="margin:0 0 8px 0;">Module sliders (0 = off, 1 = native rate)</h3>'),
    s_cross, s_nitr, s_electro, s_quorum, s_kill, s_prod,
])

display(controls)
display(output_area)
on_change()  # initial render so the user sees the default scene before touching anything


Output()